# Bike Demo
adapted from https://github.com/brightway-lca/from-the-ground-up/tree/main/try.brightway.dev

In [2]:
# imports tell us which libraries we need to run the code.
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd

## 1. Projects and Database setup

The first thing to learn about `bw2data` is the concept of projects. Each project is self-contained, and independent of other projects. Each has its own subdirectory. This can lead to data duplication, but helps keep each project safe from the changes in the others.

We start in the `default` project:

In [3]:
bd.projects.current

'default'

Let's create a new project:

In [6]:
# deleting the project so I can rerun and check everything works but will remove this before students use it
bd.projects.delete_project(name='demo', delete_dir=True)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lordj01\\AppData\\Local\\pylca\\Brightway3\\demo.fe01ce2a\\lci\\databases.db'

In [7]:
bd.projects.set_current(name='demo')
bd.projects.current

'demo'

To enter our data into BW, we need to create the nodes (or activities), and then the edges (or exchanges). We will create these nodes in a `Database`. A database in BW is just a collection of nodes - it can be large or small, there aren't any general rules.

Let's check what databases we have:

In [8]:
bd.databases

Databases dictionary with 0 objects

This should be empty, we haven't created any yet.

Let's register a new database:

In [9]:
bike_db = bd.Database("bike")
bike_db.register()
bd.databases # now when we check this, we see our new database

Databases dictionary with 1 object(s):
	bike

We wont be using ecoinvent in this example, but we can access Bioshpere3 (which is a database full of elementary flow data and is open access) and we can access the LCIA methods so let's add those (this might take a minute)

In [10]:
bi.create_default_biosphere3(overwrite=False)
bi.create_default_lcia_methods(overwrite=False, rationalize_method_names=False, shortcut=True)

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds


100%|██████████| 4709/4709 [00:00<00:00, 7733.13it/s]


Vacuuming database 
Created database: biosphere3


ValueError: Can't understand elementary flow identifier ['biosphere3', '9990b51b-7023-4700-bca0-1a32ef921f74'] in data line (['biosphere3', '9990b51b-7023-4700-bca0-1a32ef921f74'], 1.6)

Now when we look at our databases again, we can see that bioshpere3 is included

In [ ]:
bd.databases

We should make it easy to access for later

In [ ]:
bs_db = bd.Database("biosphere3")

## 2. Activities and Exchanges

Next we can start to create our foreground, in this case, a bike.

We will start with creating our nodes or activities:

In [ ]:
bike = bike_db.new_activity(
    name ='bike',
    unit ='unit',
    location = 'DK',
    type = 'product',
    code = 'bike', # codes must be unique within a database, but can be anything. See what happens if you try to save this activity again.
)
bike.save()

In [ ]:
bike_production = bike_db.new_activity(
    name ='bike production',
    location ='DK',
    type = 'process',
    code = 'bike_production',
)
bike_production.save()

Now try to create the activities for carbon fibre and carbon fibre production

In [ ]:
# note to remove the values for these so the students have to write them
cf = bike_db.new_activity(
    name = 'carbon fibre', 
    unit = 'kg',
    location = 'DE',
    type = 'product',
    code = 'cf',
)
cf.save()

cf_production = bike_db.new_activity(
    name = 'carbon fibre production',
    location = 'DE',
    type = 'process',
    code = 'cf_production',
)
cf_production.save()

In our process graph, we can see that some CO2 is also created during this step. We will include this later. For now, let's make the natural gas activities

In [ ]:
# Try to add your own activities here! 

Now we need to connect our activities together using exchanges! 

We'll start from the bike again:

In [ ]:
# This is the production exchange, which links the production of the bike to the bike activity. The amount is usually 1.
bike_production.new_exchange(
    input = bike,
    type = 'production',
    amount = 1,
).save()

# Consumption exchanges represent how much carbon fibre is used per bike production. 
bike_production.new_exchange(
    input = cf,
    type = 'consumption',
    amount = 2.5,
).save()

# Again we will need a production exchange for the carbon fibre production.
cf_production.new_exchange(
    input = cf,
    type = 'production',
    amount = 1,
).save()

Now try to continue building the exchanges based on the natural gas you have created before.
Ignore the CO2 again for now, we will add that later.

In [ ]:
# Try to make the natural gas exchanges here!

Great job! Now we can have a little look at what we've made:

In [ ]:
# Let's check our activities
for act in bike_db:
    print(act)

In [ ]:
# Now let's check out the exchanges for our carbon fibre
# You can do this by just printing the exchanges
for exc in cf_production.exchanges():
    print(exc)

In [ ]:
# Or you can use Python's built in list comprehension
[exc for exc in cf_production.exchanges()]

Oh yeah, it's missing our CO2. We should add that next

## 3. Working with larger datasets

First we need to find the CO2 that we want to use in the bioshpere3 database. Let's look for it using list comprehension.

In [ ]:
co2 = [flow for flow in bs_db if 'carbon dioxide' in flow['name'].lower()]
co2

Okay, so there's more than one! Great, we should have known it wouldn't be that simple. Let's have a bit of a closer look at one of these flows 

In [ ]:
dict(random.choice(co2))

Now we know what information is attached to these bioshpere flows, we can be more specific about what we're after

In [ ]:
co2 = [flow for flow in bs_db if 'carbon dioxide, fossil' == flow['name'].lower() and ('air',) == flow['categories']]
co2

That's better, but it's still giving us a list so we will just say we're taking the first response here

In [ ]:
co2 = co2[0]
dict(co2)

Okay, now we need to add it to our carbon fibre production

In [ ]:
cf_production.new_exchange(
    input = co2,
    type = 'biosphere', # this is a biosphere exchange
    amount = 26.6,
).save()

## 4. LCIA

If we want to know the impacts of our life cycle, we need to run them based on impact assessment methods (of which there are many). Have a look at how many there are:

In [ ]:
bd.methods

Good thing our trusty list comprehension can help us here again

In [ ]:
[m for m in bd.methods if m[0] == "ReCiPe 2016 v1.03, midpoint (H)"]

let's narrow it down further to just climate change

In [ ]:
recipe_CC = [m for m in bd.methods if 'climate change' in m[1].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[0]][0]
recipe_CC

Now we know what method we're using, we can actually figure out the impact of making our bike!

In the next cell, we will execute 4 lines of code to perform our LCA - yep, just 4 lines! Let’s get a breakdown of what is going on here:

bc.LCA({bike: 1}, recipe_CC): we create an LCA ‘object’ that requires the demand of our LCA (functional unit) and the method
lca.lci(): Calculates the life cycle inventory from technosphere and biosphere matrices based on the demand
lca.lcia(): Calculates the impact by multiplying the characterization matrix from our chosen method with the LCI matrix created in the prior line of code.
lca.score(): Returns the LCA score

In [ ]:
functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    {bike: 1},
    method=recipe_CC,
)
lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()
lca.lcia()
lca.score